# I-Beam Buckling — Shell Model + VCT (IPE200)

**Strategy used for results:** vanilla `ops.recorder("mpco", ...)` capturing displacement, reaction force, and section force at every load step + `Results.from_mpco` + viewer.

Simply-supported IPE200 (3 m span), vertical point load at midspan applied at the top flange, modelled with **ASDShellQ4** in corotational formulation. Two outputs:

1. **Static analysis** — incremental load with deformed shape captured in MPCO at every step.
2. **Linear buckling via VCT** (Vibration Correlation Technique) — track natural frequencies as load grows and extrapolate $\omega^2 \to 0$ for $P_{cr}$.

**Why a shell model?** Beam theory predicts global Euler buckling and gives closed-form LTB estimates, but cannot capture **local** buckling. A shell model captures global flexural, lateral-torsional, and local buckling simultaneously.

**VCT.** Under increasing compressive/bending stress, geometric stiffness reduces effective stiffness, lowering natural frequencies. In the linear regime
$$\omega^2(P) \approx \omega_0^2 \left(1 - \frac{P}{P_{cr}}\right)$$
Plotting $\omega^2$ vs $P$ gives a straight line whose x-intercept is $P_{cr}$.

## 1. Imports and parameters

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import openseespy.opensees as ops
import gmsh

from apeGmsh import apeGmsh, Results

# IPE200 cross-section (consistent units: mm, N, MPa, tonne)
h   = 200.0     # total depth
b   = 100.0     # flange width
tw  = 5.6       # web thickness
tf  = 8.5       # flange thickness
L   = 3000.0    # span

d = h - tf      # mid-surface flange separation

# Material — steel
E   = 210_000.0
NU  = 0.3
RHO = 7.85e-9
G   = E / (2 * (1 + NU))

# Reference / step loads (downward at midspan, on top flange)
P_REF  = 1_000.0     # initial reference load [N]
P_STEP = 5_000.0     # increment per VCT step [N]
N_STEPS = 20         # number of increments

# Output
OUT_DIR   = Path("ibeam_buckling_out")
OUT_DIR.mkdir(exist_ok=True)
MPCO_FILE = OUT_DIR / "ibeam_buckling.mpco"

# Section properties (for analytical comparison)
Iy = (b * h**3 - (b - tw) * (h - 2 * tf)**3) / 12
Iz = (2 * tf * b**3 + (h - 2 * tf) * tw**3) / 12
J  = (2 * b * tf**3 + (h - 2 * tf) * tw**3) / 3
print(f"IPE200: Iy={Iy:.0f}, Iz={Iz:.0f}, J={J:.0f} mm⁴")

## 2. Geometry — 5 mid-surface plates

Cross-section at $x=0$:

```
  p5───────p4───────p6      z = d  (top flange)
             │
             │  web
             │
  p1───────p2───────p3      z = 0  (bottom flange)
 y=-b/2    y=0    y=b/2
```

Five rectangular surfaces (2 × bottom flange halves, web, 2 × top flange halves) sharing edges at the web–flange junctions. Transfinite meshing → structured quad grid → ASDShellQ4 elements.

In [ ]:
g = apeGmsh(model_name="IPE200_Buckling", verbose=False)
g.begin()
geo = g.model.geometry

# Cross-section corner points at x = 0 and x = L
p1  = geo.add_point(0, -b/2, 0, lc=30)
p2  = geo.add_point(0,  0,   0, lc=30)
p3  = geo.add_point(0,  b/2, 0, lc=30)
p4  = geo.add_point(0,  0,   d, lc=30)
p5  = geo.add_point(0, -b/2, d, lc=30)
p6  = geo.add_point(0,  b/2, d, lc=30)

p7  = geo.add_point(L, -b/2, 0, lc=30)
p8  = geo.add_point(L,  0,   0, lc=30)
p9  = geo.add_point(L,  b/2, 0, lc=30)
p10 = geo.add_point(L,  0,   d, lc=30)
p11 = geo.add_point(L, -b/2, d, lc=30)
p12 = geo.add_point(L,  b/2, d, lc=30)

# Cross-section lines @ x = 0
c1 = geo.add_line(p1, p2)
c2 = geo.add_line(p2, p3)
c3 = geo.add_line(p2, p4)    # web
c4 = geo.add_line(p5, p4)
c5 = geo.add_line(p4, p6)

# Cross-section lines @ x = L
c6  = geo.add_line(p7,  p8)
c7  = geo.add_line(p8,  p9)
c8  = geo.add_line(p8,  p10)  # web
c9  = geo.add_line(p11, p10)
c10 = geo.add_line(p10, p12)

# Longitudinal lines
e1 = geo.add_line(p1, p7)
e2 = geo.add_line(p2, p8)
e3 = geo.add_line(p3, p9)
e4 = geo.add_line(p4, p10)
e5 = geo.add_line(p5, p11)
e6 = geo.add_line(p6, p12)

# 5 surfaces
s_bf_l = geo.add_plane_surface(geo.add_curve_loop([c1, e2, -c6, -e1]),  label="BotFlange_L")
s_bf_r = geo.add_plane_surface(geo.add_curve_loop([c2, e3, -c7, -e2]),  label="BotFlange_R")
s_web  = geo.add_plane_surface(geo.add_curve_loop([c3, e4, -c8, -e2]),  label="Web")
s_tf_l = geo.add_plane_surface(geo.add_curve_loop([-c4, e5, c9, -e4]),  label="TopFlange_L")
s_tf_r = geo.add_plane_surface(geo.add_curve_loop([c5, e6, -c10, -e4]), label="TopFlange_R")

# Physical groups for sections + supports
g.physical.add(2, [s_bf_l, s_bf_r, s_tf_l, s_tf_r], name="Flanges")
g.physical.add(2, [s_web], name="Web")
g.physical.add(1, [c1, c2, c3, c4, c5],     name="PinEnd")
g.physical.add(1, [c6, c7, c8, c9, c10],    name="RollerEnd")

## 3. Transfinite quad mesh

In [ ]:
n_half_flange = 4    # across half-flange
n_web_height  = 12   # across web height
n_length      = 60   # along beam length

for c in [c1, c2, c4, c5, c6, c7, c9, c10]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_half_flange + 1)
for c in [c3, c8]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_web_height + 1)
for c in [e1, e2, e3, e4, e5, e6]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_length + 1)

for s in [s_bf_l, s_bf_r, s_web, s_tf_l, s_tf_r]:
    gmsh.model.mesh.setTransfiniteSurface(s)
    gmsh.model.mesh.setRecombine(2, s)

g.mesh.generation.set_order(1)
g.mesh.generation.generate(2)
g.mesh.partitioning.renumber(dim=2, base=1)

fem = g.mesh.queries.get_fem_data(dim=2)
print(f"mesh built: {fem.info.n_nodes} nodes, {fem.info.n_elems} elements")

## 4. OpenSees shell model — vanilla openseespy

ASDShellQ4 with `-corotational`, two `ElasticMembranePlateSection` sections (flange / web), simply-supported fork BCs, and a single point load at the midspan top-flange node.

- **Pin end** (x = 0): fix u_x, u_y, u_z on every cross-section node.
- **Roller end** (x = L): fix u_y, u_z (free axial).

Both ends fix u_y at all section nodes — fork-support condition, standard for LTB studies.

In [ ]:
ops.wipe()
ops.model("basic", "-ndm", 3, "-ndf", 6)

# Nodes
for nid, xyz in fem.nodes.get():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

# Sections
SEC_FLANGE, SEC_WEB = 1, 2
ops.section("ElasticMembranePlateSection", SEC_FLANGE, E, NU, tf, RHO)
ops.section("ElasticMembranePlateSection", SEC_WEB,    E, NU, tw, RHO)

# Elements — ASDShellQ4 -corotational
for group in fem.elements.get(pg="Flanges"):
    for eid, conn in zip(group.ids, group.connectivity):
        ops.element(
            "ASDShellQ4", int(eid),
            int(conn[0]), int(conn[1]), int(conn[2]), int(conn[3]),
            SEC_FLANGE, "-corotational",
        )
for group in fem.elements.get(pg="Web"):
    for eid, conn in zip(group.ids, group.connectivity):
        ops.element(
            "ASDShellQ4", int(eid),
            int(conn[0]), int(conn[1]), int(conn[2]), int(conn[3]),
            SEC_WEB, "-corotational",
        )

# BCs — fork supports
for n in fem.nodes.get(pg="PinEnd").ids:
    ops.fix(int(n), 1, 1, 1, 0, 0, 0)
for n in fem.nodes.get(pg="RollerEnd").ids:
    ops.fix(int(n), 0, 1, 1, 0, 0, 0)

# Find the load node — closest to (L/2, 0, d) on the top-web junction
ids    = np.asarray([int(i) for i in fem.nodes.ids])
coords = fem.nodes.coords
target = np.array([L/2, 0.0, d])
load_node = int(ids[np.argmin(np.linalg.norm(coords - target, axis=1))])
print(f"load node: {load_node} at {coords[np.argmin(np.linalg.norm(coords - target, axis=1))]}")

# Unit-load pattern — analyse() with LoadControl(P_step) scales it.
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)
ops.load(load_node, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0)

## 5. MPCO recorder — every analysis step

Single recorder, written next to the notebook. Captures:

- **Nodes** — `displacement`, `reactionForce` (and rotational counterparts for the 6-DOF model).
- **Elements** — `force` (per-node nodal resisting forces) and `section.force` (shell stress resultants — N, M per unit length on each Gauss point).

`-T nsteps 1` writes one frame per `analyze()` call, regardless of the LoadControl time increment.

In [ ]:
if MPCO_FILE.exists():
    MPCO_FILE.unlink()

ops.recorder(
    "mpco", str(MPCO_FILE),
    "-N", "displacement", "rotation", "reactionForce", "reactionMoment",
    "-E", "force", "section.force",
    "-T", "nsteps", 1,
)

## 6. Static incremental analysis + VCT eigen sweep

We march forward in load steps and call `ops.eigen` after each step to track the lowest frequencies. The MPCO recorder captures the deformed shape at every increment.

The eigenvalue at $P=0$ is taken first; subsequent eigen() calls happen between LoadControl steps. The geometric stiffness from corotational ASDShellQ4 makes $\omega^2$ drop monotonically as the destabilising load grows.

In [ ]:
ops.constraints("Transformation")
ops.numberer("RCM")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1.0e-6, 25)
ops.algorithm("Newton")
ops.analysis("Static")

# ω² at zero load (baseline for VCT)
N_MODES = 6
ev0 = ops.eigen(N_MODES)
load_levels = [0.0]
ev_history  = [list(ev0)]

# Step 0 — static under P_REF, gives MPCO a non-trivial first frame
ops.integrator("LoadControl", P_REF)
if ops.analyze(1) != 0:
    raise RuntimeError("reference-load step failed")
ev = ops.eigen(N_MODES)
load_levels.append(P_REF)
ev_history.append(list(ev))

# Steps 1..N_STEPS — march in P_STEP increments
ops.integrator("LoadControl", P_STEP)
P_total = P_REF
print(f"{'step':>4s}  {'P [kN]':>10s}  {'f1 [Hz]':>10s}  {'f2 [Hz]':>10s}")
for k in range(N_STEPS):
    if ops.analyze(1) != 0:
        print(f"  step {k+1}: FAILED at P = {(P_total + P_STEP)/1000:.1f} kN")
        break
    P_total += P_STEP
    ev = ops.eigen(N_MODES)
    load_levels.append(P_total)
    ev_history.append(list(ev))
    f1 = np.sqrt(abs(ev[0])) / (2 * np.pi)
    f2 = np.sqrt(abs(ev[1])) / (2 * np.pi)
    print(f"{k+1:4d}  {P_total/1000:10.2f}  {f1:10.3f}  {f2:10.3f}")

load_levels = np.asarray(load_levels)
ev_history  = np.asarray(ev_history)
print(f"completed {len(load_levels)} levels up to {load_levels[-1]/1000:.1f} kN")

## 7. Flush MPCO recorder + open results

In [ ]:
ops.wipeAnalysis()
ops.remove("recorders")

results = Results.from_mpco(MPCO_FILE, fem=fem)
print(results.inspect.summary())

## 8. VCT plot — extrapolate $\omega^2 \to 0$

Linear fit $\omega^2 = \alpha P + \beta$ on the last $\sim$10 points; the x-intercept $-\beta/\alpha$ is $P_{cr}$ for that mode.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
P_cr_estimates = []

for m in range(min(4, ev_history.shape[1])):
    omega_sq = ev_history[:, m]
    ax.plot(load_levels / 1000, omega_sq, "o-", color=colors[m], ms=3, lw=1.2,
            label=f"Mode {m+1}")
    n_fit = min(10, len(load_levels))
    P_fit = load_levels[-n_fit:]
    w_fit = omega_sq[-n_fit:]
    if w_fit[-1] < w_fit[0]:
        a, c = np.polyfit(P_fit, w_fit, 1)
        if a < 0:
            P_cr = -c / a
            P_cr_estimates.append((m + 1, P_cr))
            P_ext = np.linspace(0, P_cr * 1.05, 50)
            ax.plot(P_ext / 1000, a * P_ext + c, "--", color=colors[m], lw=0.8, alpha=0.5)
            ax.axvline(P_cr / 1000, color=colors[m], ls=":", lw=0.6, alpha=0.5)

ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("P [kN]")
ax.set_ylabel(r"$\omega^2$ [rad$^2$/s$^2$]")
ax.set_title("VCT: $\\omega^2$ vs applied load")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nVCT critical load estimates (P_cr where ω² → 0):")
for mode, pcr in P_cr_estimates:
    print(f"  Mode {mode}: P_cr = {pcr/1000:.2f} kN")
if P_cr_estimates:
    P_cr_min = min(pcr for _, pcr in P_cr_estimates)
    print(f"\nLowest P_cr = {P_cr_min/1000:.2f} kN")

## 9. Analytical LTB comparison

For a simply-supported beam under a midspan point load, the elastic critical moment is
$$M_{cr} = C_1 \frac{\pi}{L}\sqrt{E I_z\, GJ}\sqrt{1 + \frac{\pi^2 E I_w}{GJ L^2}}$$
with $C_1 \approx 1.365$ for a midspan point load and $I_w \approx I_z (h - t_f)^2/4$. Then $P_{cr} = 4 M_{cr}/L$.

In [ ]:
C1 = 1.365
Iw = Iz * d**2 / 4
M_cr = C1 * (np.pi / L) * np.sqrt(E * Iz * G * J) * np.sqrt(1 + (np.pi**2 * E * Iw) / (G * J * L**2))
P_cr_analytical = 4 * M_cr / L

print(f"M_cr (analytical) = {M_cr/1e6:.3f} kN·m")
print(f"P_cr (analytical) = {P_cr_analytical/1000:.2f} kN")
if P_cr_estimates:
    P_cr_fem = min(pcr for _, pcr in P_cr_estimates)
    print(f"P_cr (FEM-VCT)    = {P_cr_fem/1000:.2f} kN")
    print(f"ratio FEM/analytical = {P_cr_fem/P_cr_analytical:.3f}")

## 10. Open the results viewer

The viewer shows the deformed beam at every captured load step. Toggle through `displacement_*`, `reaction_force_*`, and the shell `section.force` channels in the Layer stack.

In [ ]:
import os
if os.environ.get("APEGMSH_NO_VIEWER") != "1":
    results.viewer()

In [ ]:
g.end()